# 🛡️ AE-WGAN NSL-KDD — F1 par catégorie (DoS, Probe, R2L, U2R)

Même config que le notebook Table 5, mais avec évaluation **par catégorie d'attaque**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, urllib.request, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Preprocessing (Table 5)

In [ ]:
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
for f,u in {"KDDTrain+.txt":url+"KDDTrain%2B.txt","KDDTest+.txt":url+"KDDTest%2B.txt"}.items():
    if not os.path.exists(f): urllib.request.urlretrieve(u, f)

COLS = ['duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level']

ATK = {'normal':'Normal','back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS',
    'smurf':'DoS','teardrop':'DoS','mailbomb':'DoS','apache2':'DoS',
    'processtable':'DoS','udpstorm':'DoS','satan':'Probe','ipsweep':'Probe',
    'nmap':'Probe','portsweep':'Probe','mscan':'Probe','saint':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L',
    'multihop':'R2L','warezmaster':'R2L','warezclient':'R2L','spy':'R2L',
    'xlock':'R2L','xsnoop':'R2L','snmpguess':'R2L','snmpgetattack':'R2L',
    'httptunnel':'R2L','sendmail':'R2L','named':'R2L','worm':'R2L',
    'buffer_overflow':'U2R','loadmodule':'U2R','rootkit':'U2R',
    'perl':'U2R','sqlattack':'U2R','xterm':'U2R','ps':'U2R'}

FEATS = ['protocol_type','flag','land','wrong_fragment','logged_in',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_srv_rerror_rate','difficulty_level']

df_tr=pd.read_csv("KDDTrain+.txt",header=None,names=COLS)
df_te=pd.read_csv("KDDTest+.txt",header=None,names=COLS)
df_tr['label']=df_tr['label'].str.strip().str.rstrip('.')
df_te['label']=df_te['label'].str.strip().str.rstrip('.')

y_train_cat=df_tr['label'].map(ATK).fillna('Unknown')
y_test_cat=df_te['label'].map(ATK).fillna('Unknown')
y_train_detail=df_tr['label'].copy()

X_tr_df=df_tr[FEATS].copy(); X_te_df=df_te[FEATS].copy()
cat_cols=X_tr_df.select_dtypes(include=['object']).columns.tolist()
enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
X_tr_df[cat_cols]=enc.fit_transform(X_tr_df[cat_cols])
X_te_df[cat_cols]=enc.transform(X_te_df[cat_cols])
scaler=MinMaxScaler()
X_train_final=scaler.fit_transform(X_tr_df.values)
X_test_final=scaler.transform(X_te_df.values)
INPUT_DIM=24
N_NORMAL=(y_train_cat=='Normal').sum()
TARGET=int(N_NORMAL*0.10)
print(f"✅ Train:{X_train_final.shape} Test:{X_test_final.shape} Seuil:{TARGET}")
print(f"\nTrain: {dict(y_train_cat.value_counts())}")
print(f"Test:  {dict(y_test_cat.value_counts())}")

## 2. AE-WGAN (GitHub)

In [ ]:
class AEWGAN_Original:
    def __init__(self):
        self.img_rows=4;self.img_cols=6;self.channels=1
        self.img_shape=(4,6,1);self.latent_dim=32
        self.flat_dim=int(np.prod(self.img_shape))
        opt=tf.keras.optimizers.Adamax(0.00009)
        self.discriminator=self._bd()
        self.discriminator.compile(loss=self._wl,optimizer=opt,metrics=['accuracy'])
        self.generator=self._bg()
        self.encoder=self._be()
        self.discriminator.trainable=False
        z=Input(shape=(32,));img_=self.generator(z)
        img=Input(shape=(4,6,1));z_=self.encoder(img)
        self.combined=Model([z,img],[self.discriminator([z,img_]),self.discriminator([z_,img])])
        self.combined.compile(loss=[self._wl,self._wl],optimizer=opt)
    def _wl(self,yt,yp): return K.mean(yt*yp)
    def _be(self):
        m=Sequential([Flatten(input_shape=(4,6,1)),Dense(20,activation='relu'),Dense(32)])
        i=Input(shape=(4,6,1));return Model(i,m(i),name='enc')
    def _bg(self):
        m=Sequential([Dense(20,input_dim=32,activation='relu'),Dense(40,activation='relu'),
            Dense(80,activation='relu'),Dense(120,activation='relu'),Dense(160,activation='relu'),
            Dense(self.flat_dim,activation='relu'),Reshape((4,6,1))])
        z=Input(shape=(32,));return Model(z,m(z),name='gen')
    def _bd(self):
        z=Input(shape=(32,));img=Input(shape=(4,6,1))
        d=Concatenate()([z,Flatten()(img)])
        d=Dense(80,activation='relu')(d);d=Dense(40,activation='relu')(d)
        return Model([z,img],Dense(1,activation='sigmoid')(d),name='disc')
    def train(self,X,epochs=101,batch_size=32,verbose=1):
        r=X.reshape(-1,4,6,1);n=r.shape[0]
        if n<batch_size:batch_size=max(n//2,2)
        v=np.ones((batch_size,1));f=np.zeros((batch_size,1))
        for e in range(epochs):
            z=np.random.normal(0,1,(batch_size,32))
            gi=self.generator.predict(z,verbose=0)
            idx=np.random.randint(0,n,batch_size);imgs=r[idx]
            ze=self.encoder.predict(imgs,verbose=0)
            self.discriminator.train_on_batch([ze,imgs],v)
            self.discriminator.train_on_batch([z,gi],f)
            self.combined.train_on_batch([z,imgs],[v,f])
            if verbose and (e+1)%max(1,epochs//5)==0:
                print(f"      Epoch {e+1}/{epochs}")
    def generate_batch(self,n):
        return self.generator.predict(np.random.normal(0,1,(n,32)),verbose=0,batch_size=256).reshape(-1,self.flat_dim)

print("✅ AE-WGAN défini")

## 3. Oversampling (10% de Normal)

In [ ]:
print(f"Oversampling — seuil: {TARGET}")
attack_types = sorted([t for t in y_train_detail.unique() if t != 'normal'])
all_syn_X=[]; all_syn_y=[]
start=time.time()

for atk in attack_types:
    mask=y_train_detail==atk; X_a=X_train_final[mask]; n=X_a.shape[0]
    if n>=TARGET: print(f"  [{atk}] ⏩ {n}>={TARGET}"); continue
    if n<10: print(f"  [{atk}] ⏭️ {n}<10"); continue
    ng=TARGET-n
    print(f"\n  [{atk}] Training ({n} samples, +{ng})...")
    m=AEWGAN_Original()
    m.train(X_a,epochs=101,batch_size=min(32,max(n//2,2)),verbose=1)
    syn=m.generate_batch(ng)
    all_syn_X.append(syn); all_syn_y.extend([atk]*ng)
    print(f"  [{atk}] ✅ {n}+{ng}={n+ng}")

print(f"\n--- {(time.time()-start)/60:.1f} min ---")

if all_syn_X:
    Xs=np.vstack(all_syn_X); ys=pd.Series(all_syn_y)
    X_aug=np.vstack([X_train_final,Xs])
    y_aug_detail=pd.concat([y_train_detail,ys],ignore_index=True)
    y_aug_cat=y_aug_detail.map(ATK).fillna('Unknown')
else:
    X_aug=X_train_final; y_aug_cat=y_train_cat

print(f"Original: {X_train_final.shape[0]}, Augmented: {X_aug.shape[0]}")

## 4. Classifieurs + Évaluation par catégorie

In [ ]:
def build_cnn(dim,nc):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential([layers.Input(shape=(dim,1)),layers.Conv1D(40,3,activation='relu',padding='same'),
        layers.Dropout(0.1),layers.Flatten(),layers.Dense(40,activation='relu'),
        layers.Dense(out,activation=act)],name='CNN')
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def build_gru(dim,nc):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    if nc==2:
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10,activation='relu'));m.add(layers.Dense(10,activation='relu'))
    else:
        m.add(layers.GRU(30,activation='relu',return_sequences=True));m.add(layers.Dropout(0.1))
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

print("✅ Classifieurs définis")

In [ ]:
# ============================================================
# ÉVALUATION PAR CATÉGORIE (Normal, DoS, Probe, R2L, U2R)
# C'est le coeur de ce notebook
# ============================================================

def eval_per_category(clf_name, clf, X_tr, y_tr_cat, X_te, y_te_cat, input_dim):
    """
    Entraîne en multiclass (5 classes) et évalue F1 par catégorie.
    Retourne un dict avec F1 par catégorie + global.
    """
    le = LabelEncoder()
    # Fit sur toutes les classes possibles
    all_classes = sorted(list(set(y_tr_cat.unique()) | set(y_te_cat.unique())))
    le.fit(all_classes)
    y_tr = le.transform(y_tr_cat)
    y_te = le.transform(y_te_cat)
    nc = len(all_classes)

    if clf_name in ['CNN', 'GRU']:
        X_tr_3d = X_tr.reshape(-1, input_dim, 1)
        X_te_3d = X_te.reshape(-1, input_dim, 1)
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        clf.fit(X_tr_3d, y_tr, epochs=50, batch_size=256,
                validation_split=0.1, callbacks=[es], verbose=0)
        yp = clf.predict(X_te_3d, verbose=0)
        y_pred = np.argmax(yp, axis=1)
    else:
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)

    # F1 par catégorie
    results = {'classifier': clf_name}
    results['accuracy'] = accuracy_score(y_te, y_pred)
    results['f1_weighted'] = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    for cat in ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']:
        if cat in le.classes_:
            cat_idx = le.transform([cat])[0]
            # F1 binaire pour cette catégorie (one-vs-rest)
            y_te_bin = (y_te == cat_idx).astype(int)
            y_pred_bin = (y_pred == cat_idx).astype(int)
            results[f'f1_{cat}'] = f1_score(y_te_bin, y_pred_bin, zero_division=0)
            results[f'precision_{cat}'] = precision_score(y_te_bin, y_pred_bin, zero_division=0)
            results[f'recall_{cat}'] = recall_score(y_te_bin, y_pred_bin, zero_division=0)
        else:
            results[f'f1_{cat}'] = 0.0
            results[f'precision_{cat}'] = 0.0
            results[f'recall_{cat}'] = 0.0

    return results

print("✅ Évaluation par catégorie prête")

In [ ]:
# ============================================================
# RUN: Baseline + AE-WGAN, 4 classifieurs, par catégorie
# ============================================================

categories = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
classifiers_config = {
    'DT': lambda nc: DecisionTreeClassifier(max_depth=9, random_state=42),
    'RF': lambda nc: RandomForestClassifier(100, max_depth=10, random_state=42, n_jobs=-1),
    'CNN': lambda nc: build_cnn(INPUT_DIM, nc),
    'GRU': lambda nc: build_gru(INPUT_DIM, nc),
}

all_classes = sorted(list(set(y_train_cat.unique()) | set(y_test_cat.unique())))
nc = len(all_classes)

# --- BASELINE ---
print("\n" + "="*70)
print("  BASELINE — F1 par catégorie")
print("="*70)
baseline_results = []
for name, builder in classifiers_config.items():
    print(f"  Training {name}...", end=" ")
    clf = builder(nc)
    r = eval_per_category(name, clf, X_train_final, y_train_cat, X_test_final, y_test_cat, INPUT_DIM)
    baseline_results.append(r)
    print(f"Acc={r['accuracy']:.4f} | Normal={r['f1_Normal']:.3f} DoS={r['f1_DoS']:.3f} "
          f"Probe={r['f1_Probe']:.3f} R2L={r['f1_R2L']:.3f} U2R={r['f1_U2R']:.3f}")

# --- AE-WGAN ---
print("\n" + "="*70)
print("  AE-WGAN — F1 par catégorie")
print("="*70)
aewgan_results = []
for name, builder in classifiers_config.items():
    print(f"  Training {name}...", end=" ")
    clf = builder(nc)
    r = eval_per_category(name, clf, X_aug, y_aug_cat, X_test_final, y_test_cat, INPUT_DIM)
    aewgan_results.append(r)
    print(f"Acc={r['accuracy']:.4f} | Normal={r['f1_Normal']:.3f} DoS={r['f1_DoS']:.3f} "
          f"Probe={r['f1_Probe']:.3f} R2L={r['f1_R2L']:.3f} U2R={r['f1_U2R']:.3f}")

## 5. Comparaison détaillée

In [ ]:
# ============================================================
# TABLEAU COMPARATIF PAR CATÉGORIE
# ============================================================

print("\n" + "="*80)
print("  COMPARAISON DÉTAILLÉE: BASELINE vs AE-WGAN — F1 par catégorie")
print("="*80)

print(f"\n  {'Clf':<6} {'Metric':<10} {'Normal':>8} {'DoS':>8} {'Probe':>8} {'R2L':>8} {'U2R':>8} {'Global':>8}")
print(f"  {'─'*64}")

for b, e in zip(baseline_results, aewgan_results):
    name = b['classifier']
    # Baseline
    print(f"  {name:<6} {'Base F1':<10}", end="")
    for cat in categories:
        print(f" {b[f'f1_{cat}']:>7.3f}", end="")
    print(f" {b['f1_weighted']:>7.3f}")

    # AE-WGAN
    print(f"  {'':6} {'AEWGAN F1':<10}", end="")
    for cat in categories:
        print(f" {e[f'f1_{cat}']:>7.3f}", end="")
    print(f" {e['f1_weighted']:>7.3f}")

    # Delta
    print(f"  {'':6} {'Δ F1':<10}", end="")
    for cat in categories:
        d = e[f'f1_{cat}'] - b[f'f1_{cat}']
        sign = '+' if d >= 0 else ''
        color = '✅' if d > 0.01 else '❌' if d < -0.01 else '➖'
        print(f" {sign}{d:>6.3f}{color}", end="")
    dg = e['f1_weighted'] - b['f1_weighted']
    print(f" {'+' if dg>=0 else ''}{dg:>6.3f}")
    print()

In [ ]:
# ============================================================
# FOCUS R2L et U2R — Paper comparison
# ============================================================

print("\n" + "="*70)
print("  FOCUS ATTAQUES RARES: R2L et U2R")
print("="*70)

print(f"\n  Paper (CNN):")
print(f"    R2L: Baseline 0.28 → AE-WGAN 0.65 (+37%)")
print(f"    U2R: Baseline 0.09 → AE-WGAN 0.00 (-9%)")

print(f"\n  Nos résultats:")
print(f"  {'Clf':<6} {'R2L Base':>10} {'R2L AEWG':>10} {'Δ R2L':>8} {'U2R Base':>10} {'U2R AEWG':>10} {'Δ U2R':>8}")
print(f"  {'─'*60}")
for b, e in zip(baseline_results, aewgan_results):
    dr = e['f1_R2L'] - b['f1_R2L']
    du = e['f1_U2R'] - b['f1_U2R']
    print(f"  {b['classifier']:<6} {b['f1_R2L']:>10.4f} {e['f1_R2L']:>10.4f} {'+' if dr>=0 else ''}{dr:>7.4f} "
          f"{b['f1_U2R']:>10.4f} {e['f1_U2R']:>10.4f} {'+' if du>=0 else ''}{du:>7.4f}")

print(f"\n  Test set distribution:")
print(f"    R2L: {(y_test_cat=='R2L').sum()} samples")
print(f"    U2R: {(y_test_cat=='U2R').sum()} samples")

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('NSL-KDD: F1 par catégorie — Baseline vs AE-WGAN', fontsize=16, fontweight='bold')

for idx, (b, e) in enumerate(zip(baseline_results, aewgan_results)):
    ax = axes[idx//2][idx%2]
    cats = categories
    x = np.arange(len(cats))
    w = 0.35
    vals_b = [b[f'f1_{c}'] for c in cats]
    vals_e = [e[f'f1_{c}'] for c in cats]

    bars1 = ax.bar(x-w/2, vals_b, w, label='Baseline', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x+w/2, vals_e, w, label='AE-WGAN', color='#e74c3c', alpha=0.8)

    # Annotate deltas
    for i, (vb, ve) in enumerate(zip(vals_b, vals_e)):
        d = ve - vb
        if abs(d) > 0.001:
            color = 'green' if d > 0 else 'red'
            ax.annotate(f'{d:+.1%}', xy=(i+w/2, ve+0.01), ha='center', fontsize=8,
                        color=color, fontweight='bold')

    ax.set_xlabel('Catégorie')
    ax.set_ylabel('F1 Score')
    ax.set_title(f'{b["classifier"]}')
    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.legend()
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results_per_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique sauvegardé")

In [ ]:
# CSV + Drive
rows = []
for b in baseline_results:
    for cat in categories:
        rows.append({'classifier':b['classifier'],'type':'Baseline','category':cat,
                     'f1':b[f'f1_{cat}'],'precision':b[f'precision_{cat}'],'recall':b[f'recall_{cat}']})
for e in aewgan_results:
    for cat in categories:
        rows.append({'classifier':e['classifier'],'type':'AE-WGAN','category':cat,
                     'f1':e[f'f1_{cat}'],'precision':e[f'precision_{cat}'],'recall':e[f'recall_{cat}']})

df_cat = pd.DataFrame(rows)
df_cat.to_csv('results_per_category.csv', index=False)

import shutil
shutil.copy('results_per_category.csv', '/content/drive/MyDrive/')
shutil.copy('results_per_category.png', '/content/drive/MyDrive/')
print("\n✅ Sauvegardé et copié sur Drive")
print(df_cat.to_string(index=False))